# Week 4: Web Scraping + Networked Archives
### Google Colab version — updated

This notebook introduces web scraping as both a **technical method** and a **critical archival practice**.

---
**What you'll do**
- define a small scraping question
- collect a tiny dataset from a public webpage
- save a links dataset
- save an image dataset
- save a sentence dataset using keywords
- combine all datasets into one larger dataset
- generate a simple webpage to visualize the dataset

**Course connection**
- **Week 1:** erosion  
- **Week 2:** systems and dependencies  
- **Week 3:** fragments and incomplete provenance  
- **Week 4:** scraping as a way of producing new fragments

---
## 1. Framing

**Web scraping** means extracting material from a webpage, often:
- text
- links
- images
- metadata

But scraping is **never neutral**.  
It removes material from its original:
- interface
- chronology
- context
- platform logic

A scraped dataset is **not the internet itself**.  
It is a **partial reconstruction produced by a method**.

> “Found images become like textures or pigments… shorn of their historical specificity.”  
> — Ian Rothwell

Use this idea as a guide while you work: scraping does not just collect material, it **transforms** it.

---
## 2. Setup

Google Colab already runs Python in the browser, so you do **not** need to install anything locally.

Run the next two cells.

In [ ]:
# Install libraries if needed
!pip -q install beautifulsoup4 pandas requests

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from urllib.parse import urljoin, urlparse
from IPython.display import display, HTML, FileLink

---
## 3. Choose one or more source pages

For this demo, we use a short list of public pages.

You can later replace these with pages that:
- are publicly accessible
- do not require login
- are appropriate for classroom use
- allow respectful access

**Avoid** scraping private, sensitive, or restricted material.

Start small: 3–5 pages is enough for most beginner projects.

In [ ]:
urls = [
    "https://example.com",
    "https://www.iana.org/domains/example"
]

pages = []

for url in urls:
    response = requests.get(url, timeout=20)
    response.raise_for_status()
    html = response.text
    pages.append({
        "url": url,
        "html": html,
        "soup": BeautifulSoup(html, "html.parser")
    })

print("Loaded pages:")
for page in pages:
    print("-", page["url"])
    print(page["html"][:250], "\n")

---
## 4. Parse the pages

HTML is the structure underneath a webpage.

A basic multi-page scraping workflow usually looks like this:

1. define a list of URLs  
2. request each page  
3. parse the HTML  
4. locate elements on each page  
5. extract data into shared datasets  
6. save those datasets in a new form

In [ ]:
print("Page titles:")
for page in pages:
    title = page["soup"].title.get_text(strip=True) if page["soup"].title else "[no title]"
    print("-", title)

---
## 5. Extract links from multiple pages

Links make a good beginner dataset because they are easy to inspect.

In the next cell, we collect from **all pages**:
- visible link text
- original href
- absolute URL
- domain
- source page

In [ ]:
link_rows = []

for page in pages:
    source_url = page["url"]
    soup = page["soup"]

    for a in soup.find_all("a"):
        href = a.get("href")
        text = a.get_text(" ", strip=True)
        absolute_url = urljoin(source_url, href) if href else None
        link_rows.append({
            "item_type": "link",
            "label": text,
            "href": href,
            "absolute_url": absolute_url,
            "domain": urlparse(absolute_url).netloc if absolute_url else None,
            "source_page": source_url,
            "thumbnail_url": None
        })

links_df = pd.DataFrame(link_rows)
links_df.head(10)

---
## 6. Save the links dataset

In [ ]:
links_csv = "week4_links_dataset.csv"
links_df.to_csv(links_csv, index=False)
print(f"Saved: {links_csv}")

---
## 7. Extract images from multiple pages

Now we collect image information from **all pages**.

This will create a second dataset with:
- alt text
- image URL
- domain
- source page

Then we will combine it with the links dataset from above.

In [ ]:
image_rows = []

for page in pages:
    source_url = page["url"]
    soup = page["soup"]

    for img in soup.find_all("img"):
        src = img.get("src")
        alt = img.get("alt", "")
        full_src = urljoin(source_url, src) if src else None
        image_rows.append({
            "item_type": "image",
            "label": alt if alt else "[no alt text]",
            "href": src,
            "absolute_url": full_src,
            "domain": urlparse(full_src).netloc if full_src else None,
            "source_page": source_url,
            "thumbnail_url": full_src
        })

images_df = pd.DataFrame(image_rows)
images_df.head(10)

---
## 8. Save the image dataset

In [ ]:
images_csv = "week4_images_dataset.csv"
images_df.to_csv(images_csv, index=False)
print(f"Saved: {images_csv}")

---
## 9. Extract sentences from multiple pages using a list of keywords

This scraper lets you collect **sentences** from all selected pages based on keywords.

It will:
- pull paragraph text from each page
- split it into sentences
- search for sentences containing selected keywords
- save those sentences as a third dataset

This is useful if you want to treat **language as archive material**, not just links or images.

In [ ]:
import re

# Edit this list for your own project
keywords = ["example", "domain", "use"]

sentence_rows = []

for page in pages:
    source_url = page["url"]
    soup = page["soup"]

    paragraph_text = " ".join(
        p.get_text(" ", strip=True) for p in soup.find_all("p")
    )

    raw_sentences = re.split(r'(?<=[.!?])\s+', paragraph_text)

    for sentence in raw_sentences:
        cleaned = sentence.strip()
        if not cleaned:
            continue

        matched_keywords = [kw for kw in keywords if kw.lower() in cleaned.lower()]

        if matched_keywords:
            sentence_rows.append({
                "item_type": "sentence",
                "label": cleaned,
                "href": None,
                "absolute_url": None,
                "domain": urlparse(source_url).netloc,
                "source_page": source_url,
                "thumbnail_url": None,
                "matched_keywords": ", ".join(matched_keywords)
            })

sentences_df = pd.DataFrame(sentence_rows)
sentences_df.head(20)

---
## 10. Save the sentence dataset

In [ ]:
sentences_csv = "week4_sentences_dataset.csv"
sentences_df.to_csv(sentences_csv, index=False)
print(f"Saved: {sentences_csv}")

---
## 11. Add the new datasets to the preexisting datasets from before

We now combine:
- the links dataset from all pages
- the image dataset from all pages
- the sentence dataset from all pages based on keywords

This creates one larger dataset that can be visualized together.

In [ ]:
combined_df = pd.concat([links_df, images_df, sentences_df], ignore_index=True)

combined_csv = "week4_combined_dataset.csv"
combined_df.to_csv(combined_csv, index=False)

print(f"Saved combined dataset: {combined_csv}")
print("Rows in links dataset:", len(links_df))
print("Rows in image dataset:", len(images_df))
print("Rows in sentence dataset:", len(sentences_df))
print("Rows in combined dataset:", len(combined_df))

combined_df.head(20)

---
## 12. Compare pages before combining everything

Before reading the full combined dataset, it can help to compare how much material came from each page.

In [ ]:
print("Links by source page:")
print(links_df["source_page"].value_counts(), "\n")

print("Images by source page:")
print(images_df["source_page"].value_counts(), "\n")

if "source_page" in sentences_df.columns and len(sentences_df) > 0:
    print("Sentences by source page:")
    print(sentences_df["source_page"].value_counts())
else:
    print("No sentence rows found.")

---
## 12. Read the combined dataset critically

Pause and ask:
- What appears more often: links, images, or sentences?
- What kinds of labels are missing?
- What context was removed by putting these into one table?
- What new patterns were created by combining them?

In [ ]:
print(combined_df["item_type"].value_counts(dropna=False))
print("\nDomains:")
print(combined_df["domain"].dropna().value_counts())

---
## 13. Create a simple webpage to visualize the dataset

This step generates an HTML file you can open in a browser.

The page will:
- show each item as a card
- label whether it is a link, image, or sentence
- display thumbnails for image items when available
- link out to the source URL where relevant

In [ ]:
cards = []

for _, row in combined_df.iterrows():
    label = str(row.get("label", "") or "")
    item_type = str(row.get("item_type", "") or "")
    absolute_url = str(row.get("absolute_url", "") or "")
    source_page = str(row.get("source_page", "") or "")
    thumb = row.get("thumbnail_url", None)

    image_html = ""
    if pd.notna(thumb) and thumb:
        image_html = f'<img src="{thumb}" alt="{label}" class="thumb">'

    card_html = f'''
    <div class="card">
      <div class="meta">{item_type}</div>
      {image_html}
      <div class="label">{label}</div>
      <div class="small"><strong>source page:</strong> <a href="{source_page}" target="_blank">{source_page}</a></div>
      <div class="small"><strong>item url:</strong> <a href="{absolute_url}" target="_blank">{absolute_url}</a></div>
    </div>
    '''
    cards.append(card_html)

html_page = f'''
<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>Week 4 Dataset Visualization</title>
  <style>
    body {{
      font-family: Arial, sans-serif;
      margin: 40px;
      background: #f7f7f7;
      color: #111;
    }}
    h1 {{
      font-size: 28px;
      margin-bottom: 8px;
    }}
    p {{
      max-width: 800px;
      line-height: 1.5;
    }}
    .grid {{
      display: grid;
      grid-template-columns: repeat(auto-fill, minmax(280px, 1fr));
      gap: 18px;
      margin-top: 28px;
    }}
    .card {{
      background: white;
      border: 1px solid #ddd;
      border-radius: 12px;
      padding: 14px;
      box-shadow: 0 2px 6px rgba(0,0,0,0.05);
      overflow-wrap: anywhere;
    }}
    .meta {{
      font-size: 12px;
      text-transform: uppercase;
      letter-spacing: 0.08em;
      color: #666;
      margin-bottom: 8px;
    }}
    .label {{
      font-weight: 600;
      margin: 10px 0;
    }}
    .small {{
      font-size: 12px;
      color: #444;
      margin-top: 6px;
    }}
    .thumb {{
      width: 100%;
      max-height: 180px;
      object-fit: cover;
      border-radius: 8px;
      border: 1px solid #eee;
    }}
  </style>
</head>
<body>
  <h1>Week 4 Dataset Visualization</h1>
  <p>
    This page visualizes a combined dataset made from scraped links and images.
    It does not preserve the original webpage interface. It reconstructs part of the page as a new archive.
  </p>
  <div class="grid">
    {"".join(cards)}
  </div>
</body>
</html>
'''

html_filename = "week4_dataset_visualization.html"
with open(html_filename, "w", encoding="utf-8") as f:
    f.write(html_page)

print(f"Saved webpage: {html_filename}")

---
## 14. Preview the webpage inside Colab

In [ ]:
display(HTML(filename="week4_dataset_visualization.html"))

---
## 15. Download your outputs

The next cell makes clickable download links for:
- links dataset
- image dataset
- sentence dataset
- combined dataset
- webpage visualization

In [ ]:
display(FileLink(links_csv))
display(FileLink(images_csv))
display(FileLink(sentences_csv))
display(FileLink(combined_csv))
display(FileLink("week4_dataset_visualization.html"))

---
## 16. Assignment workflow

For your assignment:
1. choose 1–5 public pages
2. define one collection rule
3. collect 20–100 items total
4. save your links dataset
5. save your image dataset if relevant
6. save your sentence dataset if relevant
7. combine your datasets
8. generate a webpage to visualize them
9. write a short reflection

---
## 17. Reflection prompt

Write 300–500 words responding to these questions:

- What did you collect across the different pages?
- What did your method exclude?
- What changed when you combined the links, image, and sentence datasets?
- What differences emerged between pages?
- What is missing from the webpage visualization?
- Is this dataset an archive, a fragment, or a new artwork?

In [ ]:
reflection = '''
Write your reflection here.
'''
print(reflection)

---
## 18. What this notebook cannot capture

A scraped dataset and a generated webpage usually do **not** preserve:
- the original interface
- user interaction
- temporal change
- recommendation systems
- platform behavior
- the full meaning of context

This is why scraped archives are useful—but always incomplete.

---
## 19. Final takeaway

You are **not preserving the web as a whole**.  
You are creating a **partial, constructed archive** from it.

The datasets and the webpage are all new forms of representation.